# Meixner Distribution for Equity Risk Modeling — Interactive Demo

This notebook walks through:
1. Loading equity return data
2. Fitting the Meixner distribution via MLE
3. Comparing the fit against a Normal distribution
4. Computing rolling Value-at-Risk (VaR)
5. Running the Kupiec Proportion of Failures (POF) backtest

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, skew, kurtosis

from src.meixner_distribution import meixner, fit_meixner_mle, meixner_theoretical_moments
from src.var_calculation import rolling_meixner_var, normal_var, meixner_var
from src.kupiec_backtest import kupiec_pof_test, kupiec_traffic_light

%matplotlib inline

## 1. Load Data

Replace this with your own returns series (e.g. log returns of an equity, index, or FX rate).

In [ ]:
df = pd.read_csv('../data/synthetic_equity_returns.csv', index_col=0, parse_dates=True)
returns = df['return'].dropna().values

print(f"Observations: {len(returns)}")
print(f"Mean: {np.mean(returns):.6f}")
print(f"Std Dev: {np.std(returns):.6f}")
print(f"Skewness: {skew(returns):.4f}")
print(f"Excess Kurtosis: {kurtosis(returns):.4f}")

df['return'].plot(figsize=(12, 4), title='Daily Returns', lw=0.7)
plt.show()

## 2. Fit the Meixner Distribution via MLE

In [ ]:
fit_result = fit_meixner_mle(returns)
alpha, beta, delta, m = fit_result['params']

print(f"alpha = {alpha:.6f}")
print(f"beta  = {beta:.6f}")
print(f"delta = {delta:.6f}")
print(f"m     = {m:.6f}")
print(f"Log-Likelihood = {fit_result['loglik']:.2f}")
print(f"AIC = {fit_result['aic']:.2f}, BIC = {fit_result['bic']:.2f}")
print(f"Converged: {fit_result['success']}")

In [ ]:
moments = meixner_theoretical_moments(alpha, beta, delta, m)
pd.DataFrame({
    'Empirical': [np.mean(returns), skew(returns), kurtosis(returns)],
    'Meixner-implied': [moments['mean'], moments['skewness'], moments['excess_kurtosis']]
}, index=['Mean', 'Skewness', 'Excess Kurtosis'])

## 3. Compare Meixner vs Normal Fit

In [ ]:
mu_norm, sigma_norm = np.mean(returns), np.std(returns, ddof=1)
x_grid = np.linspace(returns.min() * 1.1, returns.max() * 1.1, 1000)

meixner_pdf_vals = meixner.pdf(x_grid, alpha, beta, delta, m)
normal_pdf_vals = norm.pdf(x_grid, mu_norm, sigma_norm)

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(returns, bins=80, density=True, alpha=0.4, label='Empirical')
ax.plot(x_grid, meixner_pdf_vals, 'r-', lw=2, label='Meixner Fit')
ax.plot(x_grid, normal_pdf_vals, 'k--', lw=2, label='Normal Fit')
ax.set_yscale('log')
ax.set_title('Tail Comparison (Log Scale)')
ax.legend()
plt.show()

## 4. Rolling 99% VaR

In [ ]:
confidence_level = 0.99
window = 250

meixner_result = rolling_meixner_var(returns, window=window,
                                      confidence_level=confidence_level,
                                      refit_every=20)
meixner_var_series = meixner_result['var_estimates']
normal_var_series = normal_var(returns, window=window, confidence_level=confidence_level)

valid = ~np.isnan(meixner_var_series)
dates = df.index[valid]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(dates, returns[valid], color='black', lw=0.6, label='Returns')
ax.plot(dates, -meixner_var_series[valid], color='crimson', label='Meixner 99% VaR')
ax.plot(dates, -normal_var_series[valid], color='gray', linestyle='--', label='Normal 99% VaR')
ax.legend()
ax.set_title('Rolling 99% VaR: Meixner vs Normal')
plt.show()

## 5. Kupiec POF Backtest

In [ ]:
test_returns = returns[valid]
meixner_kupiec = kupiec_pof_test(test_returns, meixner_var_series[valid], confidence_level)
normal_kupiec = kupiec_pof_test(test_returns, normal_var_series[valid], confidence_level)

pd.DataFrame({
    'Meixner': meixner_kupiec,
    'Normal': normal_kupiec
})

## 6. Basel Traffic Light (last 250 days)

In [ ]:
n_recent = 250
recent_returns = test_returns[-n_recent:]
recent_meixner_var = meixner_var_series[valid][-n_recent:]
recent_normal_var = normal_var_series[valid][-n_recent:]

meixner_exceptions = int(np.sum(recent_returns < -np.abs(recent_meixner_var)))
normal_exceptions = int(np.sum(recent_returns < -np.abs(recent_normal_var)))

print('Meixner:', kupiec_traffic_light(n_recent, meixner_exceptions, confidence_level))
print('Normal: ', kupiec_traffic_light(n_recent, normal_exceptions, confidence_level))